# 两个总体均值的假设检验（独立样本）完整教程：从手算到 scipy

## 📚 教学目标
1. 理解独立两样本 t 检验的前提条件与适用场景
2. 掌握合并方差（pooled variance）和分开方差（Welch's t）两种方法
3. 手算 t 统计量的每一步计算过程
4. 用 `scipy.stats.ttest_ind` 验证手算结果
5. 理解方差齐性检验（Levene 检验）的重要性及其对方法选择的影响

**参考书**：《基础统计学(第14版)》（Triola）第 9-2 章

**教学策略**：先用极小数据集（每组 10-12 个数据）让你「看见」每一步计算，再扩展到真实规模（200-500 个数据）

## 1. 场景设定：两个教学方法的比较

### 🎯 问题

一所学校想要比较两种数学教学方法对学生期末成绩的影响。随机将学生分成两组：
- **方法 A**（传统讲授法）：12 名学生
- **方法 B**（互动探究法）：15 名学生

学期结束后，收集两组的期末考试成绩，我们想知道：**两种教学方法的效果是否有显著差异？**

### 📖 书中的例子

> 书中使用了 1988 年与 2012 年美国陆军男性人员身高数据进行比较：
> - 1988 年：n₁=12, x̄₁=1739.4mm, s₁=66.6mm
> - 2012 年：n₂=15, x̄₂=1777.8mm, s₂=47.87mm

这是一个典型的**独立两样本**问题——两组数据来自不同的个体，互不相关。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# 设置中文字体和样式
import matplotlib.font_manager as fm
_cn_fonts = [f.name for f in fm.fontManager.ttflist if any(kw in f.name for kw in ['Hei', 'Song', 'PingFang', 'Arial Unicode', 'Noto Sans CJK', 'SimHei', 'Microsoft YaHei'])]
plt.rcParams['font.sans-serif'] = _cn_fonts[:3] + ['DejaVu Sans'] if _cn_fonts else ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

## 2. 独立样本 vs 配对样本的区别

在使用两样本 t 检验之前，必须先判断数据是**独立样本**还是**配对样本**。选错方法会导致完全错误的结论。

### 📐 核心区别

| 特性 | 独立样本 (Independent) | 配对样本 (Paired) |
|------|----------------------|------------------|
| **数据关系** | 两组来自不同个体 | 同一对象测量两次，或一一配对 |
| **例子** | 男身高 vs 女身高 | 夫妻身高 |
| **例子** | 药物组 vs 安慰剂组 | 服药前 vs 服药后 |
| **样本量** | n₁ 和 n₂ 可以不同 | 两组样本量必须相同 |
| **分析方法** | 两独立样本 t 检验 | 配对 t 检验（差值的单样本 t） |
| **自由度** | 取决于方差情况 | df = n - 1 |

### 💡 判断口诀

> **「同一人测两次 → 配对；两组各测一次 → 独立」**

本教程聚焦于**独立样本**的情形。

## 3. 独立两样本 t 检验的条件

### 📐 前提条件

使用独立两样本 t 检验前，需要满足以下条件：

1. **独立性**：两个样本是独立的简单随机样本
2. **正态性**：两个总体都服从正态分布（或样本量足够大：$n_1 > 30$ 且 $n_2 > 30$）
3. **未知方差**：$\sigma_1$ 和 $\sigma_2$ 未知

### 💡 关于方差的两种情况

书中介绍了两种处理方式：

| 情况 | 假设 | 方法 | 自由度 |
|------|------|------|--------|
| **方差不相等** | $\sigma_1 \neq \sigma_2$（或未知） | Welch's t 检验 | Welch-Satterthwaite 公式 |
| **方差相等** | $\sigma_1 = \sigma_2$ | 合并方差 t 检验 | $df = n_1 + n_2 - 2$ |

实际操作中，先用 **Levene 检验**判断方差是否相等，再选择对应的方法。

## 4. 微型数据：手算每一步

### 📐 检验统计量公式（方差未知且不相等 — Welch's t）

$$t = \frac{(\bar{x}_1 - \bar{x}_2) - (\mu_1 - \mu_2)}{\sqrt{\frac{s_1^2}{n_1} + \frac{s_2^2}{n_2}}}$$

其中：
- $\bar{x}_1, \bar{x}_2$ = 两个样本的均值
- $\mu_1 - \mu_2$ = 假设的总体均值差（通常为 0）
- $s_1^2, s_2^2$ = 两个样本的方差
- $n_1, n_2$ = 两个样本的样本量

### 📐 自由度（保守估计）

$$df = \min(n_1 - 1, \, n_2 - 1)$$

### 📐 自由度（Welch-Satterthwaite 公式，公式 9-1）

$$df = \frac{(A + B)^2}{\frac{A^2}{n_1 - 1} + \frac{B^2}{n_2 - 1}}$$

其中 $A = \dfrac{s_1^2}{n_1}$，$B = \dfrac{s_2^2}{n_2}$

### 📝 模拟数据

我们模拟两组学生成绩数据：
- 方法 A（传统）：12 名学生，总体均值约 72 分
- 方法 B（互动）：15 名学生，总体均值约 78 分

In [ ]:
# ========== 生成微型数据集 ==========
# --- 数据生成过程 (DGP) ---
# 方法 A: 正态分布, μ=72, σ=8, n=12
# 方法 B: 正态分布, μ=78, σ=6, n=15

group_a = np.array([68, 75, 70, 73, 65, 78, 71, 69, 74, 66, 72, 70])
group_b = np.array([80, 76, 82, 78, 75, 79, 81, 77, 83, 74, 78, 76, 80, 79, 77])

n1 = len(group_a)
n2 = len(group_b)

print("📊 两组数据：")
print(f"  方法 A (传统): n₁ = {n1}, 数据 = {group_a}")
print(f"  方法 B (互动): n₂ = {n2}, 数据 = {group_b}")

In [ ]:
# ========== 📊 步骤 1: 计算基本统计量 ==========

x1_bar = np.mean(group_a)
x2_bar = np.mean(group_b)
s1_sq = np.var(group_a, ddof=1)  # 样本方差（无偏）
s2_sq = np.var(group_b, ddof=1)
s1 = np.std(group_a, ddof=1)
s2 = np.std(group_b, ddof=1)

print("📊 步骤 1: 计算基本统计量")
print(f"  方法 A: n₁ = {n1}, x̄₁ = {x1_bar:.4f}, s₁² = {s1_sq:.4f}, s₁ = {s1:.4f}")
print(f"  方法 B: n₂ = {n2}, x̄₂ = {x2_bar:.4f}, s₂² = {s2_sq:.4f}, s₂ = {s2:.4f}")
print(f"\n  样本均值差: x̄₁ - x̄₂ = {x1_bar - x2_bar:.4f}")
print(f"\n💡 方法 B 的均值比方法 A 高 {x2_bar - x1_bar:.1f} 分，但这是真实差异还是抽样误差？")

In [ ]:
# ========== 📊 步骤 2: 计算标准误 ==========

se = np.sqrt(s1_sq / n1 + s2_sq / n2)

print("📊 步骤 2: 计算标准误 (Standard Error)")
print(f"  SE = √(s₁²/n₁ + s₂²/n₂)")
print(f"     = √({s1_sq:.4f}/{n1} + {s2_sq:.4f}/{n2})")
print(f"     = √({s1_sq/n1:.4f} + {s2_sq/n2:.4f})")
print(f"     = √{s1_sq/n1 + s2_sq/n2:.4f}")
print(f"     = {se:.4f}")
print(f"\n💡 标准误衡量的是两个样本均值之差的抽样变异性")

In [ ]:
# ========== 📊 步骤 3: 计算 t 统计量 ==========

# H₀: μ₁ - μ₂ = 0（两种方法效果相同）
mu_diff_0 = 0
t_stat = ((x1_bar - x2_bar) - mu_diff_0) / se

print("📊 步骤 3: 计算 t 统计量")
print(f"  H₀: μ₁ - μ₂ = 0")
print(f"  H₁: μ₁ - μ₂ ≠ 0（双尾检验）")
print(f"\n  t = ((x̄₁ - x̄₂) - 0) / SE")
print(f"    = ({x1_bar:.4f} - {x2_bar:.4f} - 0) / {se:.4f}")
print(f"    = {x1_bar - x2_bar:.4f} / {se:.4f}")
print(f"    = {t_stat:.4f}")
print(f"\n💡 t 值为 {t_stat:.4f}，表示两个样本均值差偏离 0 约 {abs(t_stat):.2f} 个标准误")

In [ ]:
# ========== 📊 步骤 4: 计算自由度（两种方法） ==========

# 方法一：保守估计
df_conservative = min(n1 - 1, n2 - 1)

# 方法二：Welch-Satterthwaite 公式
A = s1_sq / n1
B = s2_sq / n2
df_welch = (A + B) ** 2 / (A**2 / (n1 - 1) + B**2 / (n2 - 1))

print("📊 步骤 4: 计算自由度")
print(f"\n  方法一 — 保守估计:")
print(f"    df = min(n₁-1, n₂-1) = min({n1-1}, {n2-1}) = {df_conservative}")
print(f"\n  方法二 — Welch-Satterthwaite 公式 (公式 9-1):")
print(f"    A = s₁²/n₁ = {s1_sq:.4f}/{n1} = {A:.4f}")
print(f"    B = s₂²/n₂ = {s2_sq:.4f}/{n2} = {B:.4f}")
print(f"    df = (A+B)² / (A²/(n₁-1) + B²/(n₂-1))")
print(f"       = ({A+B:.4f})² / ({A**2:.6f}/{n1-1} + {B**2:.6f}/{n2-1})")
print(f"       = {(A+B)**2:.6f} / {A**2/(n1-1) + B**2/(n2-1):.6f}")
print(f"       = {df_welch:.4f}")
print(f"\n💡 Welch 公式给出更精确的自由度（通常不是整数），保守法取较小值更安全")

In [ ]:
# ========== 📊 步骤 5: 计算 p 值并做出决策 ==========

alpha = 0.05

# 使用 Welch 自由度
p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df=df_welch))

# 临界值
t_critical = stats.t.ppf(1 - alpha / 2, df=df_welch)

print("📊 步骤 5: 假设检验决策")
print(f"  显著性水平 α = {alpha}")
print(f"  t 统计量 = {t_stat:.4f}")
print(f"  自由度 (Welch) = {df_welch:.4f}")
print(f"  临界值 ±t_{{α/2}} = ±{t_critical:.4f}")
print(f"  p 值 = {p_value:.6f}")
print(f"\n  判定: |t| = {abs(t_stat):.4f} {'>' if abs(t_stat) > t_critical else '<='} t_{{α/2}} = {t_critical:.4f}")
print(f"  判定: p = {p_value:.6f} {'<' if p_value < alpha else '>='} α = {alpha}")

if p_value < alpha:
    print(f"\n  💡 结论: 拒绝 H₀。两种教学方法的成绩有显著差异。")
else:
    print(f"\n  💡 结论: 不拒绝 H₀。没有足够证据表明两种教学方法有显著差异。")

## 5. 用 scipy.stats.ttest_ind 验证

### 🔬 scipy 的两种模式

`scipy.stats.ttest_ind` 通过 `equal_var` 参数控制使用哪种方法：

| 参数 | 方法 | 对应公式 |
|------|------|----------|
| `equal_var=True` | 合并方差 t 检验 | $s_p^2 = \frac{(n_1-1)s_1^2 + (n_2-1)s_2^2}{n_1+n_2-2}$ |
| `equal_var=False` | Welch's t 检验（默认推荐） | Welch-Satterthwaite 自由度 |

In [ ]:
# ========== 用 scipy 验证（Welch's t） ==========

# Welch's t 检验（equal_var=False，推荐）
t_welch, p_welch = stats.ttest_ind(group_a, group_b, equal_var=False)

print("🔬 scipy.stats.ttest_ind 验证 (Welch's t):")
print(f"  手算 t 统计量 = {t_stat:.6f}")
print(f"  scipy t 统计量 = {t_welch:.6f}")
print(f"  手算 p 值     = {p_value:.6f}")
print(f"  scipy p 值    = {p_welch:.6f}")
print(f"\n  ✅ 验证通过！手算结果与 scipy 完全一致。")

## 6. 合并方差 vs 分开方差（Welch's t）

### 📐 合并方差 t 检验（假设 $\sigma_1 = \sigma_2$）

当两个总体方差相等时，可以将两个样本方差「合并」为一个更精确的估计：

$$s_p^2 = \frac{(n_1 - 1)s_1^2 + (n_2 - 1)s_2^2}{n_1 + n_2 - 2}$$

此时 t 统计量为：

$$t = \frac{(\bar{x}_1 - \bar{x}_2) - (\mu_1 - \mu_2)}{s_p \sqrt{\frac{1}{n_1} + \frac{1}{n_2}}}$$

自由度为：

$$df = n_1 + n_2 - 2$$

### 💡 两种方法对比

| 特性 | 合并方差 t | Welch's t |
|------|-----------|----------|
| **假设** | $\sigma_1 = \sigma_2$ | 无此假设 |
| **自由度** | $n_1 + n_2 - 2$（整数） | Welch-Satterthwaite（通常非整数） |
| **适用性** | 方差相等时更精确 | 更通用、更安全 |
| **scipy 参数** | `equal_var=True` | `equal_var=False` |

In [ ]:
# ========== 合并方差 t 检验 ==========

# 计算合并方差
s_p_sq = ((n1 - 1) * s1_sq + (n2 - 1) * s2_sq) / (n1 + n2 - 2)
s_p = np.sqrt(s_p_sq)
df_pooled = n1 + n2 - 2

# 合并方差 t 统计量
se_pooled = s_p * np.sqrt(1 / n1 + 1 / n2)
t_pooled = ((x1_bar - x2_bar) - 0) / se_pooled
p_pooled = 2 * (1 - stats.t.cdf(abs(t_pooled), df=df_pooled))

print("📊 合并方差 t 检验（手算）")
print(f"  s_p² = ((n₁-1)s₁² + (n₂-1)s₂²) / (n₁+n₂-2)")
print(f"       = ({n1-1}×{s1_sq:.4f} + {n2-1}×{s2_sq:.4f}) / ({n1}+{n2}-2)")
print(f"       = {(n1-1)*s1_sq:.4f} + {(n2-1)*s2_sq:.4f}) / {df_pooled}")
print(f"       = {s_p_sq:.4f}")
print(f"  s_p = {s_p:.4f}")
print(f"\n  SE(pooled) = s_p × √(1/n₁ + 1/n₂) = {s_p:.4f} × √(1/{n1} + 1/{n2}) = {se_pooled:.4f}")
print(f"  t = ({x1_bar:.4f} - {x2_bar:.4f}) / {se_pooled:.4f} = {t_pooled:.4f}")
print(f"  df = {n1} + {n2} - 2 = {df_pooled}")
print(f"  p = {p_pooled:.6f}")

# scipy 验证
t_pooled_scipy, p_pooled_scipy = stats.ttest_ind(group_a, group_b, equal_var=True)

print(f"\n🔬 scipy (equal_var=True) 验证:")
print(f"  手算 t  = {t_pooled:.6f}")
print(f"  scipy t = {t_pooled_scipy:.6f}")
print(f"  手算 p  = {p_pooled:.6f}")
print(f"  scipy p = {p_pooled_scipy:.6f}")
print(f"\n  ✅ 验证通过！")

# 对比两种方法
print(f"\n📝 两种方法对比:")
print(f"  {'方法':<20} {'t 统计量':>10} {'自由度':>10} {'p 值':>12}")
print(f"  {'-'*54}")
print(f"  {'Welch-s t':<20} {t_stat:>10.4f} {df_welch:>10.4f} {p_value:>12.6f}")
print(f"  {'合并方差 t':<20} {t_pooled:>10.4f} {df_pooled:>10.1f} {p_pooled:>12.6f}")
print(f"\n💡 两种方法的结论一致，但 t 值和 p 值略有不同")

## 7. 方差齐性检验（Levene 检验）

### 📐 Levene 检验

在选择合并方差还是 Welch's t 之前，先检验两个总体方差是否相等：

- $H_0$: $\sigma_1^2 = \sigma_2^2$（方差相等）
- $H_1$: $\sigma_1^2 \neq \sigma_2^2$（方差不相等）

### 💡 决策逻辑

```
Levene 检验 p > α → 方差相等 → 合并方差 t 检验 (equal_var=True)
Levene 检验 p ≤ α → 方差不等 → Welch's t 检验 (equal_var=False)
```

In [ ]:
# ========== 方差齐性检验 ==========

levene_stat, levene_p = stats.levene(group_a, group_b)

print("📊 Levene 方差齐性检验")
print(f"  H₀: σ₁² = σ₂²（方差相等）")
print(f"  H₁: σ₁² ≠ σ₂²（方差不相等）")
print(f"\n  Levene 统计量 = {levene_stat:.4f}")
print(f"  p 值 = {levene_p:.6f}")
print(f"  显著性水平 α = {alpha}")

if levene_p > alpha:
    print(f"\n  💡 结论: p > α，不拒绝 H₀，认为方差相等")
    print(f"  → 推荐使用: 合并方差 t 检验 (equal_var=True)")
else:
    print(f"\n  💡 结论: p ≤ α，拒绝 H₀，认为方差不等")
    print(f"  → 推荐使用: Welch's t 检验 (equal_var=False)")

## 8. 置信区间构建

### 📐 两总体均值差的置信区间

$$(\bar{x}_1 - \bar{x}_2) - E < (\mu_1 - \mu_2) < (\bar{x}_1 - \bar{x}_2) + E$$

其中误差范围为：

$$E = t_{\alpha/2} \times \sqrt{\frac{s_1^2}{n_1} + \frac{s_2^2}{n_2}}$$

这里 $t_{\alpha/2}$ 是自由度为 Welch-Satterthwaite 值的 t 分布临界值。

### 💡 解读

> 置信区间告诉我们：两个总体均值差的**真实值**有多大可能落在某个范围内。
> 如果置信区间**不包含 0**，则与拒绝 H₀ 等价。

In [ ]:
# ========== 构建 95% 置信区间 ==========

confidence_level = 0.95
alpha_ci = 1 - confidence_level

# 临界值
t_crit = stats.t.ppf(1 - alpha_ci / 2, df=df_welch)

# 误差范围
E = t_crit * se

# 置信区间
diff_means = x1_bar - x2_bar
ci_lower = diff_means - E
ci_upper = diff_means + E

print(f"📊 步骤 6: 构建 {confidence_level*100:.0f}% 置信区间")
print(f"  样本均值差: x̄₁ - x̄₂ = {diff_means:.4f}")
print(f"  临界值 t_{{α/2}} (df={df_welch:.2f}) = {t_crit:.4f}")
print(f"  标准误 SE = {se:.4f}")
print(f"  误差范围 E = t_{{α/2}} × SE = {t_crit:.4f} × {se:.4f} = {E:.4f}")
print(f"\n  置信区间:")
print(f"    ({diff_means:.4f} - {E:.4f}, {diff_means:.4f} + {E:.4f})")
print(f"    = ({ci_lower:.4f}, {ci_upper:.4f})")

print(f"\n💡 解读:")
if ci_lower > 0 or ci_upper < 0:
    print(f"  置信区间不包含 0，说明两总体均值有显著差异")
    print(f"  我们有 {confidence_level*100:.0f}% 的信心认为 μ₁-μ₂ 在 ({ci_lower:.2f}, {ci_upper:.2f}) 之间")
else:
    print(f"  置信区间包含 0，不能断定两总体均值有显著差异")

## 9. 可视化：t 分布、箱线图与置信区间

### 📊 三幅图的说明

1. **t 分布图**：展示 t 统计量在分布中的位置，标记拒绝域
2. **箱线图**：直观对比两组数据的分布
3. **置信区间图**：展示均值差的置信区间是否跨越 0

In [ ]:
# ========== 可视化：三合一图 ==========

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- 图 1: t 分布与检验统计量 ---
ax1 = axes[0]
x_t = np.linspace(-5, 5, 1000)
y_t = stats.t.pdf(x_t, df=df_welch)

ax1.plot(x_t, y_t, 'b-', linewidth=2, label=f"t distribution (df={df_welch:.1f})")
ax1.fill_between(x_t, 0, y_t, alpha=0.1, color='steelblue')

# 拒绝域
x_reject_left = x_t[x_t < -t_critical]
y_reject_left = stats.t.pdf(x_reject_left, df=df_welch)
x_reject_right = x_t[x_t > t_critical]
y_reject_right = stats.t.pdf(x_reject_right, df=df_welch)

ax1.fill_between(x_reject_left, 0, y_reject_left, alpha=0.3, color='#e74c3c', label='Rejection Region')
ax1.fill_between(x_reject_right, 0, y_reject_right, alpha=0.3, color='#e74c3c')

# 标记 t 统计量
ax1.axvline(x=t_stat, color='#e74c3c', linestyle='--', linewidth=2, label=f't = {t_stat:.4f}')
ax1.axvline(x=-t_critical, color='gray', linestyle=':', alpha=0.7)
ax1.axvline(x=t_critical, color='gray', linestyle=':', alpha=0.7)

ax1.set_xlabel('T Value', fontsize=12)
ax1.set_ylabel('Probability Density', fontsize=12)
ax1.set_title('t Distribution with Test Statistic', fontsize=14, fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

# --- 图 2: 箱线图对比 ---
ax2 = axes[1]
bp = ax2.boxplot([group_a, group_b], labels=['Method A\n(Traditional)', 'Method B\n(Interactive)'],
                 patch_artist=True, widths=0.5)
colors_box = ['#2ecc71', 'steelblue']
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

# 添加数据点
for i, (data, x_pos) in enumerate(zip([group_a, group_b], [1, 2])):
    jitter = np.random.normal(0, 0.03, len(data))
    ax2.scatter([x_pos] * len(data) + jitter, data, alpha=0.5, color='black', s=20, zorder=5)

ax2.set_ylabel('Score', fontsize=12)
ax2.set_title('Score Comparison: Box Plot', fontsize=14, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

# --- 图 3: 置信区间图 ---
ax3 = axes[2]
ax3.errorbar(0, diff_means, yerr=E, fmt='o', color='steelblue', markersize=10,
             capsize=10, capthick=2, linewidth=2, label='95% CI')
ax3.axhline(y=0, color='#e74c3c', linestyle='--', linewidth=1.5, alpha=0.7, label='H₀: μ₁-μ₂ = 0')

# 填充置信区间
ax3.fill_between([-0.3, 0.3], ci_lower, ci_upper, alpha=0.2, color='steelblue')

ax3.set_xlim(-0.5, 0.5)
ax3.set_xlabel('', fontsize=12)
ax3.set_ylabel('Difference in Means (μ₁ - μ₂)', fontsize=12)
ax3.set_title('95% Confidence Interval\nfor Difference in Means', fontsize=14, fontweight='bold')
ax3.set_xticks([0])
ax3.set_xticklabels(['Method A - Method B'])
ax3.legend(fontsize=10)
ax3.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  图1：t 统计量 {t_stat:.4f} 落入拒绝域，表明差异在统计上显著")
print(f"  图2：方法 B 的成绩整体高于方法 A，中位数差异明显")
print(f"  图3：95% 置信区间 ({ci_lower:.2f}, {ci_upper:.2f}) 不包含 0，确认差异显著")

## 10. 大样本扩展（200-500 数据点）

### 📐 大样本时的行为

当 $n_1 > 30$ 且 $n_2 > 30$ 时：
- t 分布近似于标准正态分布
- 正态性假设的重要性降低（中心极限定理）
- 自由度较大，t 临界值接近 z 临界值

### 📝 模拟参数

- 方法 A：n₁ = 250, μ = 72, σ = 10
- 方法 B：n₂ = 300, μ = 75, σ = 12

In [ ]:
# ========== 大样本数据生成 ==========
# --- 数据生成过程 (DGP) ---
# 方法 A: 正态分布, μ=72, σ=10, n=250
# 方法 B: 正态分布, μ=75, σ=12, n=300

n1_large = 250
n2_large = 300
group_a_large = np.random.normal(loc=72, scale=10, size=n1_large)
group_b_large = np.random.normal(loc=75, scale=12, size=n2_large)

print(f"📊 大样本数据概况：")
print(f"  方法 A: n₁ = {n1_large}, x̄₁ = {np.mean(group_a_large):.2f}, s₁ = {np.std(group_a_large, ddof=1):.2f}")
print(f"  方法 B: n₂ = {n2_large}, x̄₂ = {np.mean(group_b_large):.2f}, s₂ = {np.std(group_b_large, ddof=1):.2f}")

In [ ]:
# ========== 大样本完整分析 ==========

# Levene 检验
lev_stat_l, lev_p_l = stats.levene(group_a_large, group_b_large)
print(f"📊 Levene 方差齐性检验:")
print(f"  统计量 = {lev_stat_l:.4f}, p = {lev_p_l:.6f}")
use_pooled_l = lev_p_l > alpha
print(f"  结论: 方差{'相等' if use_pooled_l else '不等'}，使用{'合并方差' if use_pooled_l else 'Welch-s'} t 检验")

# t 检验
t_welch_l, p_welch_l = stats.ttest_ind(group_a_large, group_b_large, equal_var=False)
t_pool_l, p_pool_l = stats.ttest_ind(group_a_large, group_b_large, equal_var=True)

print(f"\n📊 t 检验结果:")
print(f"  {'方法':<20} {'t 统计量':>10} {'p 值':>12}")
print(f"  {'-'*44}")
print(f"  {'Welch-s t':<20} {t_welch_l:>10.4f} {p_welch_l:>12.6f}")
print(f"  {'合并方差 t':<20} {t_pool_l:>10.4f} {p_pool_l:>12.6f}")

# 置信区间
x1_bar_l = np.mean(group_a_large)
x2_bar_l = np.mean(group_b_large)
s1_sq_l = np.var(group_a_large, ddof=1)
s2_sq_l = np.var(group_b_large, ddof=1)
se_l = np.sqrt(s1_sq_l / n1_large + s2_sq_l / n2_large)
A_l = s1_sq_l / n1_large
B_l = s2_sq_l / n2_large
df_welch_l = (A_l + B_l)**2 / (A_l**2/(n1_large-1) + B_l**2/(n2_large-1))
t_crit_l = stats.t.ppf(0.975, df=df_welch_l)
E_l = t_crit_l * se_l
diff_l = x1_bar_l - x2_bar_l

print(f"\n📊 95% 置信区间:")
print(f"  均值差 = {diff_l:.4f}")
print(f"  临界值 t = {t_crit_l:.4f} (df = {df_welch_l:.2f})")
print(f"  置信区间 = ({diff_l - E_l:.4f}, {diff_l + E_l:.4f})")

if p_welch_l < alpha:
    print(f"\n💡 结论: p = {p_welch_l:.6f} < α = {alpha}，拒绝 H₀，两组成绩有显著差异")
else:
    print(f"\n💡 结论: p = {p_welch_l:.6f} ≥ α = {alpha}，不拒绝 H₀")

print(f"\n💡 大样本时，Welch's t 和合并方差 t 的结果非常接近")
print(f"  自由度 df = {df_welch_l:.2f}（接近正态分布的 z 值）")

In [ ]:
# ========== 大样本可视化 ==========

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 图 1: 直方图对比 ---
ax1 = axes[0]
ax1.hist(group_a_large, bins=25, density=True, alpha=0.6, color='#2ecc71', edgecolor='white',
         label=f'Method A (n={n1_large})')
ax1.hist(group_b_large, bins=25, density=True, alpha=0.6, color='steelblue', edgecolor='white',
         label=f'Method B (n={n2_large})')

# 叠加理论正态曲线
x_range = np.linspace(40, 110, 300)
y_a = stats.norm.pdf(x_range, np.mean(group_a_large), np.std(group_a_large, ddof=1))
y_b = stats.norm.pdf(x_range, np.mean(group_b_large), np.std(group_b_large, ddof=1))
ax1.plot(x_range, y_a, '-', color='#2ecc71', linewidth=2)
ax1.plot(x_range, y_b, '-', color='steelblue', linewidth=2)

ax1.set_xlabel('Score', fontsize=12)
ax1.set_ylabel('Density', fontsize=12)
ax1.set_title('Score Distribution: Method A vs Method B', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)

# --- 图 2: 置信区间对比（微型 vs 大样本） ---
ax2 = axes[1]

# 微型数据 CI
ax2.errorbar(1, diff_means, yerr=E, fmt='s', color='#2ecc71', markersize=10,
             capsize=10, capthick=2, linewidth=2, label=f'Micro (n={n1}+{n2})')

# 大样本 CI
ax2.errorbar(2, diff_l, yerr=E_l, fmt='o', color='steelblue', markersize=10,
             capsize=10, capthick=2, linewidth=2, label=f'Large (n={n1_large}+{n2_large})')

ax2.axhline(y=0, color='#e74c3c', linestyle='--', linewidth=1.5, alpha=0.7, label='H₀: μ₁-μ₂ = 0')
ax2.set_xlim(0.5, 2.5)
ax2.set_xticks([1, 2])
ax2.set_xticklabels([f'Micro\n(n={n1}+{n2})', f'Large\n(n={n1_large}+{n2_large})'])
ax2.set_ylabel('Difference in Means (μ₁ - μ₂)', fontsize=12)
ax2.set_title('Confidence Interval Comparison', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  图1：大样本下两组分布更加平滑，均值差异清晰可见")
print(f"  图2：大样本的置信区间更窄，说明估计更精确")
print(f"    微型数据 CI 宽度: {2*E:.2f}")
print(f"    大样本 CI 宽度: {2*E_l:.2f}")

## 11. 结果报告模板

### 📝 学术论文中的标准写法

在论文中报告独立两样本 t 检验结果时，需要包含以下信息：

1. 使用的检验方法（合并方差或 Welch's t）
2. 自由度
3. t 统计量
4. p 值
5. 效应量（Cohen's d）
6. 置信区间

### 📐 效应量 Cohen's d

$$d = \frac{\bar{x}_1 - \bar{x}_2}{s_p}$$

其中 $s_p$ 是合并标准差。解读标准：
- $|d| < 0.2$：微小效应
- $0.2 \leq |d| < 0.5$：小效应
- $0.5 \leq |d| < 0.8$：中等效应
- $|d| \geq 0.8$：大效应

In [ ]:
# ========== 结果报告模板 ==========

# Cohen's d 效应量
s_pooled_all = np.sqrt(((n1 - 1) * s1_sq + (n2 - 1) * s2_sq) / (n1 + n2 - 2))
cohens_d = abs(x1_bar - x2_bar) / s_pooled_all

# 效应量解读
if cohens_d < 0.2:
    effect_size_label = "微小 (negligible)"
elif cohens_d < 0.5:
    effect_size_label = "小 (small)"
elif cohens_d < 0.8:
    effect_size_label = "中等 (medium)"
else:
    effect_size_label = "大 (large)"

print("=" * 60)
print("📋 独立两样本 t 检验结果报告")
print("=" * 60)

print(f"\n🎯 研究问题:")
print(f"   两种教学方法（传统 vs 互动）对学生成绩是否有显著影响？")

print(f"\n📊 数据概况:")
print(f"   方法 A (传统): n₁ = {n1}, x̄₁ = {x1_bar:.2f}, s₁ = {s1:.2f}")
print(f"   方法 B (互动): n₂ = {n2}, x̄₂ = {x2_bar:.2f}, s₂ = {s2:.2f}")

print(f"\n🔬 方差齐性检验:")
print(f"   Levene 检验: F = {levene_stat:.4f}, p = {levene_p:.6f}")
print(f"   结论: 方差{'相等' if levene_p > alpha else '不等'}")

print(f"\n🧮 统计检验:")
print(f"   方法: {'合并方差 t 检验' if levene_p > alpha else 'Welch-s t 检验'}")
print(f"   t({df_welch:.2f}) = {t_stat:.4f}")
print(f"   p = {p_value:.6f}")
print(f"   Cohen's d = {cohens_d:.4f} ({effect_size_label})")

print(f"\n📏 95% 置信区间:")
print(f"   μ₁ - μ₂ ∈ ({ci_lower:.4f}, {ci_upper:.4f})")

print(f"\n🎯 结论:")
if p_value < alpha:
    print(f"   ✓ 在 α = {alpha} 的显著性水平下，拒绝 H₀")
    print(f"   ✓ 两种教学方法的成绩有统计学上的显著差异")
    print(f"   ✓ t({df_welch:.2f}) = {t_stat:.4f}, p = {p_value:.6f}, d = {cohens_d:.4f}")
else:
    print(f"   ✗ 在 α = {alpha} 的显著性水平下，不拒绝 H₀")
    print(f"   ✗ 没有足够证据表明两种教学方法有显著差异")

print("\n" + "=" * 60)

## 12. 📌 核心概念回顾

### 📌 独立两样本 t 检验 (Independent Two-Sample t-Test)
- **定义**: 用于比较两个独立总体的均值是否存在显著差异的假设检验方法
- **公式**: $t = \frac{(\bar{x}_1 - \bar{x}_2) - (\mu_1 - \mu_2)}{\sqrt{\frac{s_1^2}{n_1} + \frac{s_2^2}{n_2}}}$
- **含义**: t 值衡量的是两个样本均值之差相对于抽样变异性的大小
- **判断标准**: |t| > t 临界值 或 p < α 时拒绝 H₀

### 📌 合并方差 (Pooled Variance)
- **定义**: 当两个总体方差相等时，将两个样本方差加权平均得到的方差估计
- **公式**: $s_p^2 = \frac{(n_1-1)s_1^2 + (n_2-1)s_2^2}{n_1+n_2-2}$
- **含义**: 利用了两个样本的信息，提供更精确的方差估计
- **判断标准**: 仅在 Levene 检验表明方差相等时使用

### 📌 Welch's t 检验 (Welch's t-Test)
- **定义**: 不假设两个总体方差相等的独立两样本 t 检验
- **公式**: 自由度使用 Welch-Satterthwaite 公式: $df = \frac{(A+B)^2}{A^2/(n_1-1) + B^2/(n_2-1)}$
- **含义**: 更通用、更稳健的方法，推荐作为默认选择
- **判断标准**: 适用于方差未知且可能不相等的情况

### 📌 方差齐性检验 (Levene's Test)
- **定义**: 检验两个或多个总体方差是否相等的统计检验
- **公式**: $H_0: \sigma_1^2 = \sigma_2^2$
- **含义**: 决定使用合并方差 t 还是 Welch's t 的依据
- **判断标准**: p > α 时认为方差相等，使用合并方差 t

### 📌 效应量 Cohen's d
- **定义**: 衡量两个均值之间差异大小的标准化指标
- **公式**: $d = \frac{\bar{x}_1 - \bar{x}_2}{s_p}$
- **含义**: 不受样本量影响的差异大小度量
- **判断标准**: |d| < 0.2 微小, 0.2-0.5 小, 0.5-0.8 中, ≥ 0.8 大

### 🔗 完整流程
```
检查前提条件 → 方差齐性检验 → 选择方法 → 计算 t 统计量 → 查找 p 值 → 做出决策 → 计算效应量
     ↓              ↓             ↓            ↓             ↓           ↓            ↓
  独立性/正态性   Levene 检验   合并/Welch    t = ...       p = ...    拒绝/不拒绝   Cohen's d
```

### 📝 检验指标汇总

| 指标 | 合并方差 t | Welch's t |
|------|-----------|----------|
| **假设** | $\sigma_1 = \sigma_2$ | 无方差假设 |
| **自由度** | $n_1 + n_2 - 2$ | Welch-Satterthwaite |
| **标准误** | $s_p\sqrt{1/n_1+1/n_2}$ | $\sqrt{s_1^2/n_1+s_2^2/n_2}$ |
| **适用场景** | 方差相等时 | 通用（推荐默认） |
| **scipy 参数** | `equal_var=True` | `equal_var=False` |

## 13. ❌ 常见误区

### ❌ 误区 1: 独立样本和配对样本混用
**✓ 正确理解**: 独立样本是两组不同个体的数据（如男女身高），配对样本是同一对象的两次测量（如前后测）。选错方法会得到完全错误的结论。判断关键：数据是否一一对应。

### ❌ 误区 2: 不检验方差齐性就直接用合并方差 t 检验
**✓ 正确做法**: 应先用 Levene 检验判断方差是否相等。如果方差不等却使用合并方差 t，会导致 I 类错误率偏离设定的 α 水平。实际操作中，推荐默认使用 Welch's t（`equal_var=False`），它在方差相等时表现也很好。

### ❌ 误区 3: p 值不显著就「证明」了没有差异
**✓ 正确理解**: 「不拒绝 H₀」不等于「接受 H₀」。p ≥ α 只是说没有足够证据证明差异存在，而不是证明差异不存在。可能是样本量不足（检验效能低）。应报告置信区间，让读者看到效应大小的不确定性范围。

### ❌ 误区 4: 统计显著等于实际重要
**✓ 正确理解**: 统计显著性（p < α）只说明差异不太可能由抽样误差导致，但不代表差异有实际意义。大样本下，即使微小的差异也会统计显著。应结合效应量（Cohen's d）和置信区间综合判断。例如 d = 0.05 的差异虽然 p < 0.001，但实际意义可能很小。

### ❌ 误区 5: 样本量不等就不能做两样本 t 检验
**✓ 正确理解**: 两样本 t 检验**不要求** $n_1 = n_2$。样本量不等时，Welch's t 检验依然有效。实际上，样本量不等时 Welch's t 比合并方差 t 更稳健。公式中的 $n_1$ 和 $n_2$ 可以不同。